# 11 · Guardar y recuperar trayectorias

Usaremos la capa de entrada/salida para serializar una trayectoria a JSON y reconstruirla posteriormente.

> **ProteinMotorSim 0.1.1** · Ejemplo educativo. Las tasas predeterminadas del modelo son ilustrativas y no representan todavía una parametrización experimental validada.

## Preparación del entorno

Esta celda localiza automáticamente la raíz del repositorio e incorpora `src/` a la ruta de importación. Instala antes las dependencias con `python -m pip install -e ".[dev,viz]"` y selecciona el kernel `Python (ProteinMotorSim)`.

In [1]:
from pathlib import Path
import sys

# Busca la raíz del proyecto desde el directorio actual o sus carpetas padre.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in _candidates if (path / "pyproject.toml").exists() and (path / "src").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "No se encontró ProteinMotorSim. Abre JupyterLab desde la carpeta raíz "
        "del proyecto o desde su carpeta notebooks/."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Proyecto detectado: {PROJECT_ROOT}")

Proyecto detectado: /home/zix/ProteinMotorSim


In [2]:
from pathlib import Path

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.io.serialization import save_trajectory, load_trajectory

trajectory = GillespieSimulator().run(
    Kinesin1(), Environment(atp_concentration=1.0e-3),
    t_max=0.3, seed=101,
)
output_dir = PROJECT_ROOT / "results"
output_file = output_dir / "trajectory_example.json"
save_trajectory(trajectory, output_file)
print("Archivo guardado en:", output_file)

Archivo guardado en: /home/zix/ProteinMotorSim/results/trajectory_example.json


## 1. Revisar el archivo JSON

In [3]:
print(output_file.read_text(encoding="utf-8")[:1200])

{
  "times_s": [
    0.0,
    5.811284004153596e-05,
    0.003079552675060882,
    0.015279638266598986,
    0.015419640968166138,
    0.015758662067950995,
    0.017916240184665054,
    0.01866885835907935,
    0.020054856582840598,
    0.02943957925440626,
    0.02986724582020051,
    0.03832237207929935,
    0.06721754040940782,
    0.0673907133446081,
    0.07243407315312028,
    0.072739504944889,
    0.07414544551106027,
    0.07470714580798911,
    0.08464240583437355,
    0.08530038864068998,
    0.08949563144438417,
    0.0971973859626672,
    0.09790644966249554,
    0.12750416957713,
    0.14279878559406037,
    0.14451778818729755,
    0.15083287578541882,
    0.15861791160009206,
    0.16197267184294128,
    0.16222071717378603,
    0.16318726713958331,
    0.16357986282026354,
    0.198358887116049,
    0.19939521271558264,
    0.19980793966685942,
    0.21508211610321015,
    0.22236474903539458,
    0.22315379985301698,
    0.23593933188092484,
    0.23753266705561685,


## 2. Cargar la trayectoria

In [4]:
loaded = load_trajectory(output_file)
print("Registros originales:", len(trajectory.times_s))
print("Registros cargados:   ", len(loaded.times_s))
print("Posiciones idénticas: ", trajectory.positions_nm == loaded.positions_nm)
print("Estados idénticos:    ", trajectory.states == loaded.states)
print("Eventos idénticos:    ", trajectory.events == loaded.events)
print("ATP original/cargado:", trajectory.atp_consumed_total, loaded.atp_consumed_total)

Registros originales: 46
Registros cargados:    46
Posiciones idénticas:  True
Estados idénticos:     True
Eventos idénticos:     True
ATP original/cargado: 15 15


## 3. Nota sobre persistencia

El formato JSON permite inspeccionar los datos con herramientas externas. En la versión actual se guarda la trayectoria y su consumo total de ATP; todavía no se almacena automáticamente la configuración completa del motor, ambiente, parámetros y versión del simulador. Esa trazabilidad se añadirá más adelante.